<a href="https://colab.research.google.com/github/EbinJosephM/EbinJoseph-Artificial-retina-paper/blob/main/electro_app_V_fromkaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio cellpose trackpy tifffile scikit-image

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
"""
Electrotaxis Cell Tracking — Gradio App (Colab version)
=========================================
Wraps the Cellpose + trackpy electrotaxis pipeline (segmentation, linking,
quality filtering, β/γ metrics, population stats) in a browser UI.

Colab edits vs. the HF Spaces version:
  1. load_images accepts both path strings and file objects (Gradio version-proof)
  2. demo.launch(share=True, debug=True) at the bottom, no __main__ guard
"""

import os
import shutil
import tempfile
import zipfile

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import sem
from tifffile import imread
from skimage.measure import regionprops, find_contours
import trackpy as tp
import gradio as gr

tp.quiet()

# ZeroGPU support: only present when running on a HF Space with ZeroGPU.
try:
    import spaces
    HAS_ZEROGPU = True
except ImportError:
    HAS_ZEROGPU = False


# ----------------------------------------------------------------------
# GPU-heavy step, isolated so the @spaces.GPU decorator (if available)
# only wraps the part that actually needs the GPU.
# ----------------------------------------------------------------------
def _segment_impl(images, use_gpu):
    from cellpose import models
    model = models.CellposeModel(gpu=use_gpu)
    masks, flows, styles = model.eval(list(images), diameter=None)
    return masks


if HAS_ZEROGPU:
    _segment_impl = spaces.GPU(duration=180)(_segment_impl)


def segment_images(images, use_gpu):
    try:
        return _segment_impl(images, use_gpu)
    except RuntimeError as e:
        if use_gpu and "no kernel image" in str(e).lower():
            # incompatible GPU (e.g. older card) -> fall back to CPU
            return _segment_impl(images, False)
        raise


# ----------------------------------------------------------------------
# Pipeline steps (mirrors the notebook, condensed into functions)
# ----------------------------------------------------------------------
def load_images(files):
    """files: list of Gradio file objects OR path strings (depends on the
    Gradio version). Gradio saves each upload into its own randomly-named
    temp subdirectory, so sorting by the FULL path would sort by that random
    prefix, not by filename -- silently scrambling frame order. Sort by
    basename instead, which matches how the original notebook reads a folder
    of TIFFs."""
    paths = sorted([f if isinstance(f, str) else f.name for f in files],
                   key=lambda p: os.path.basename(p))
    frame_names = [os.path.basename(p) for p in paths]
    images = []
    for p in paths:
        img = imread(p)
        if img.ndim == 3:
            img = img[..., 0]
        images.append(img)
    return images, frame_names


def build_detections(masks, edge_margin, h, w):
    rows = []
    for frame_idx, mask in enumerate(masks):
        for prop in regionprops(mask):
            y, x = prop.centroid
            min_row, min_col, max_row, max_col = prop.bbox
            touches_edge = (
                min_row <= edge_margin or min_col <= edge_margin or
                max_row >= h - edge_margin or max_col >= w - edge_margin
            )
            rows.append({
                "frame": frame_idx, "x": x, "y": y,
                "area": prop.area, "touches_edge": touches_edge,
                "label": prop.label,  # raw Cellpose label for this frame --
                                       # needed to recover this exact cell's
                                       # pixel mask later (e.g. BMAC analysis)
            })
    return pd.DataFrame(rows)


def link_trajectories(detections, search_range, memory):
    linked = tp.link(detections, search_range=search_range, memory=memory)
    return linked.rename(columns={"particle": "cell_id"})


def quality_filter(linked, min_len, max_jump, min_ratio):
    def has_issue(track):
        track = track.sort_values("frame")
        area_ratio = track["area"] / track["area"].shift(1)
        divides_or_dies = ((area_ratio > max_jump) | (area_ratio < min_ratio)).any()
        touches_edge = track["touches_edge"].any()
        return divides_or_dies, touches_edge

    keep_ids, drop_reasons = [], {}
    for cell_id, track in linked.groupby("cell_id"):
        if len(track) < min_len:
            drop_reasons[cell_id] = "too_short"
            continue
        divides_or_dies, touches_edge = has_issue(track)
        if divides_or_dies:
            drop_reasons[cell_id] = "division_rounding_or_death"
            continue
        if touches_edge:
            drop_reasons[cell_id] = "touches_edge"
            continue
        keep_ids.append(cell_id)

    filtered = linked[linked["cell_id"].isin(keep_ids)].copy().sort_values(["cell_id", "frame"])
    return filtered, drop_reasons


def mean_sem_n(series):
    s = series.dropna()
    if len(s) > 1:
        return s.mean(), sem(s), len(s)
    return (s.mean() if len(s) else np.nan), np.nan, len(s)


def circ_moments(theta_array):
    """First and second circular trigonometric moments (mean cos, sin,
    cos2, sin2) of an array of angles (radians), as used for the beta/gamma
    directional-cosine parameters in Sroka et al. 2018 (Methods Mol Biol
    1749, ch.23). NaNs are dropped before averaging."""
    theta_array = np.asarray(theta_array, dtype=float)
    theta_array = theta_array[~np.isnan(theta_array)]
    if len(theta_array) == 0:
        return np.nan, np.nan, np.nan, np.nan
    cos1 = np.mean(np.cos(theta_array))
    sin1 = np.mean(np.sin(theta_array))
    cos2 = np.mean(np.cos(2 * theta_array))
    sin2 = np.mean(np.sin(2 * theta_array))
    return cos1, cos2, sin1, sin2


def _wrap_angle(a):
    """Wrap angle(s) in radians to (-pi, pi]."""
    return (np.asarray(a) + np.pi) % (2 * np.pi) - np.pi


# ----------------------------------------------------------------------
# Main pipeline, called by the Gradio button
# ----------------------------------------------------------------------
def run_pipeline(
    files,
    frame_interval_min, pixel_to_um,
    field_x, field_y,
    search_range_px, memory_frames,
    min_track_len, edge_margin_px,
    max_area_jump, min_area_ratio,
    require_complete, use_gpu,
    progress=gr.Progress(),
):
    if not files:
        raise gr.Error("Upload the TIFF frames of your movie first.")

    workdir = tempfile.mkdtemp(prefix="electrotaxis_")

    progress(0.05, desc="Loading images")
    images, frame_names = load_images(files)
    n_frames = len(images)
    h, w = images[0].shape[:2]

    progress(0.15, desc="Segmenting cells (Cellpose)")
    masks = segment_images(images, use_gpu)

    progress(0.45, desc="Building detections table")
    detections = build_detections(masks, edge_margin_px, h, w)
    if len(detections) == 0:
        raise gr.Error("No cells were segmented — check the images or try lowering thresholds.")

    progress(0.55, desc="Linking trajectories (trackpy)")
    linked = link_trajectories(detections, search_range_px, memory_frames)

    progress(0.62, desc="Applying quality filters")
    filtered, drop_reasons = quality_filter(linked, min_track_len, max_area_jump, min_area_ratio)
    if filtered["cell_id"].nunique() == 0:
        raise gr.Error("All trajectories were filtered out — loosen the quality thresholds and retry.")

    filtered["is_complete"] = filtered.groupby("cell_id")["frame"].transform(
        lambda f: f.nunique() == n_frames
    )

    # Stable, human-friendly numbering for every kept cell ("Cell 1", "Cell 2",
    # ...), so the same number can be used later to pick a cell for BMAC
    # field-reversal analysis without having to re-identify it manually.
    cell_id_to_number = {
        cid: i + 1 for i, cid in enumerate(sorted(filtered["cell_id"].unique()))
    }
    filtered["cell_number"] = filtered["cell_id"].map(cell_id_to_number)

    # calibration + relative coords
    filtered["time_min"] = filtered["frame"] * frame_interval_min
    filtered["x_um"] = filtered["x"] * pixel_to_um
    filtered["y_um"] = -filtered["y"] * pixel_to_um
    filtered["x0"] = filtered.groupby("cell_id")["x_um"].transform("first")
    filtered["y0"] = filtered.groupby("cell_id")["y_um"].transform("first")
    filtered["x_rel"] = filtered["x_um"] - filtered["x0"]
    filtered["y_rel"] = filtered["y_um"] - filtered["y0"]
    filtered["dx"] = filtered.groupby("cell_id")["x_um"].diff()
    filtered["dy"] = filtered.groupby("cell_id")["y_um"].diff()
    filtered["step_length_um"] = np.sqrt(filtered["dx"] ** 2 + filtered["dy"] ** 2)

    field_vec = np.array([field_x, field_y], dtype=float)
    norm = np.linalg.norm(field_vec)
    field_vec = field_vec / norm if norm > 0 else np.array([1.0, 0.0])
    field_angle = np.arctan2(field_vec[1], field_vec[0])

    progress(0.72, desc="Computing per-cell trajectory metrics")
    records = []
    for cell_id, track in filtered.groupby("cell_id"):
        track = track.sort_values("frame")
        is_complete = bool(track["is_complete"].iloc[0])

        # Total migration distance = length of the path (sum of step lengths)
        total_distance_um = track["step_length_um"].sum(skipna=True)
        duration_min = track["time_min"].iloc[-1] - track["time_min"].iloc[0]

        # Net displacement: initial position -> final position
        x_um = track["x_um"].values
        y_um = track["y_um"].values
        dx_net = x_um[-1] - x_um[0]
        dy_net = y_um[-1] - y_um[0]
        displacement_um = float(np.hypot(dx_net, dy_net))

        # Angle between the displacement vector and the applied field,
        # in degrees, wrapped to (-180, 180]. 0 = along the field,
        # +/-180 = against it.
        if displacement_um > 0:
            theta = float(_wrap_angle(np.arctan2(dy_net, dx_net) - field_angle))
            angle_deg = float(np.degrees(theta))
            cos_angle = float(np.cos(theta))
        else:
            angle_deg = cos_angle = np.nan
        # Displacement component along the field axis (negative = against)
        disp_along_field_um = displacement_um * cos_angle if displacement_um > 0 else 0.0

        velocity = displacement_um / duration_min if duration_min > 0 else np.nan
        speed = total_distance_um / duration_min if duration_min > 0 else np.nan

        records.append({
            "cell_id": cell_id, "cell_number": cell_id_to_number[cell_id],
            "is_complete": is_complete,
            "n_frames_tracked": len(track), "duration_min": duration_min,
            "Displacement_um": displacement_um,                 # initial -> final
            "Velocity_um_min": velocity,                        # displacement / time
            "Angle_deg": angle_deg,                             # vs. applied field
            "Cos_angle": cos_angle,
            "Speed_um_min": speed,                              # path length / time
            "Displacement_along_field_um": disp_along_field_um,
            "Total_distance_um": total_distance_um,             # path length
        })
    summary = pd.DataFrame(records)

    progress(0.82, desc="Computing population statistics")
    # Averages are mean +/- SEM ACROSS CELLS (n = number of kept cells).
    # Angle_deg uses the CIRCULAR mean (an arithmetic mean of angles is
    # misleading near +/-180 deg), so it has no SEM.
    pop_param_cols = [
        "Displacement_um", "Velocity_um_min", "Angle_deg", "Cos_angle",
        "Speed_um_min", "Displacement_along_field_um", "Total_distance_um",
    ]
    subset_df = summary[summary["is_complete"]] if require_complete else summary
    pop_stats = {}
    for col in pop_param_cols:
        if col == "Angle_deg":
            a = np.radians(subset_df[col].dropna().to_numpy())
            m = (float(np.degrees(np.arctan2(np.mean(np.sin(a)), np.mean(np.cos(a)))))
                 if len(a) else np.nan)
            pop_stats[col] = {"mean": m, "sem": np.nan, "n": len(a)}
        else:
            m, se, n = mean_sem_n(subset_df[col])
            pop_stats[col] = {"mean": m, "sem": se, "n": n}
    pop_stats_df = pd.DataFrame(pop_stats).T.reset_index().rename(columns={"index": "parameter"})

    progress(0.9, desc="Rendering plots")

    # QC image
    fig_qc, ax_qc = plt.subplots(figsize=(6, 6))
    ax_qc.imshow(images[0], cmap="gray")
    kept0 = filtered[filtered["frame"] == filtered["frame"].min()]
    ax_qc.scatter(kept0["x"], kept0["y"], s=50, facecolors="none",
                  edgecolors="yellow", linewidths=1.5)
    for _, row in kept0.iterrows():
        ax_qc.annotate(
            str(int(row["cell_number"])), (row["x"], row["y"]),
            xytext=(4, 4), textcoords="offset points",
            color="yellow", fontsize=7, fontweight="bold",
        )
    ax_qc.set_title(f"Cells included (frame 0), n={kept0['cell_id'].nunique()}")
    ax_qc.axis("off")
    qc_path = os.path.join(workdir, "qc_selected_cells_frame0.png")
    fig_qc.savefig(qc_path, dpi=200, bbox_inches="tight")
    plt.close(fig_qc)

    # trajectory plot
    fig_traj, ax_traj = plt.subplots(figsize=(5.5, 5.5))
    for cell_id, track in filtered.groupby("cell_id"):
        ax_traj.plot(track["x_rel"], track["y_rel"], linewidth=0.5, color="black")
    ax_traj.set_aspect("equal")
    ax_traj.spines["left"].set_position("zero")
    ax_traj.spines["bottom"].set_position("zero")
    ax_traj.spines["right"].set_color("none")
    ax_traj.spines["top"].set_color("none")
    ax_traj.set_title("Trajectories (relative coordinates, µm)")
    traj_path = os.path.join(workdir, "cell_trajectories_relative.png")
    fig_traj.savefig(traj_path, dpi=200, bbox_inches="tight")
    plt.close(fig_traj)

    # polar plot
    filtered["distance_um"] = np.hypot(filtered["x_rel"], filtered["y_rel"])
    filtered["angle_rad"] = np.arctan2(filtered["y_rel"], filtered["x_rel"])
    fig_polar = plt.figure(figsize=(5.5, 5.5))
    ax_polar = plt.subplot(111, polar=True)
    for cell_id, track in filtered.groupby("cell_id"):
        ax_polar.plot(track["angle_rad"], track["distance_um"], linewidth=0.5, color="black")
    ax_polar.set_title("Distance from origin vs. direction")
    polar_path = os.path.join(workdir, "cell_polar_plot.png")
    fig_polar.savefig(polar_path, dpi=200, bbox_inches="tight")
    plt.close(fig_polar)

    # speed boxplot
    speed_vals = subset_df["Speed_um_min"].dropna()
    fig_box, ax_box = plt.subplots(figsize=(4, 5))
    if len(speed_vals) > 0:
        ax_box.boxplot(speed_vals, vert=True, showmeans=True)
        mv, sv = speed_vals.mean(), (sem(speed_vals) if len(speed_vals) > 1 else np.nan)
        ax_box.text(1.1, mv, f"Mean = {mv:.2f}\nSEM = {sv:.2f}", va="center")
    ax_box.set_ylabel("Speed, path-based (µm/min)")
    ax_box.set_title("Cell speed distribution")
    box_path = os.path.join(workdir, "cell_speed_boxplot.png")
    fig_box.savefig(box_path, dpi=200, bbox_inches="tight")
    plt.close(fig_box)

    progress(0.93, desc="Building drop-reason report")
    # Surface *why* each non-kept trajectory was dropped, so counts can be
    # sanity-checked against a manual count instead of only reporting the
    # final "kept" number.
    reason_order = ["too_short", "division_rounding_or_death", "touches_edge"]
    reason_labels = {
        "too_short": "Too short (< min track length)",
        "division_rounding_or_death": "Area jump (possible division / merge / death)",
        "touches_edge": "Touches frame edge",
    }
    reason_counts = pd.Series(drop_reasons).value_counts() if drop_reasons else pd.Series(dtype=int)
    drop_reasons_df = pd.DataFrame([
        {"reason": reason_labels[r], "n_trajectories": int(reason_counts.get(r, 0))}
        for r in reason_order
    ])
    drop_reasons_df.loc[len(drop_reasons_df)] = [
        "Kept", int(filtered["cell_id"].nunique())
    ]

    progress(0.96, desc="Rendering full-movie overlay")
    # Every frame annotated with this cell's outline, centroid, ID, and
    # trailing trajectory, so doubtful cases (merges, gaps, jumps) can be
    # checked by eye across the whole movie instead of only at frame 0.
    overlay_frames_dir = os.path.join(workdir, "movie_overlay_frames")
    os.makedirs(overlay_frames_dir, exist_ok=True)
    overlay_frame_paths = render_movie_overlay(
        images, masks, filtered, overlay_frames_dir,
    )
    overlay_gif_path = os.path.join(workdir, "movie_overlay.gif")
    try:
        from PIL import Image
        overlay_ims = [Image.open(p) for p in overlay_frame_paths]
        if overlay_ims:
            # duration is per-frame in ms; 200ms/frame is a reasonable default
            # scrub speed for checking tracking by eye
            overlay_ims[0].save(
                overlay_gif_path, save_all=True, append_images=overlay_ims[1:],
                duration=200, loop=0,
            )
    except Exception:
        overlay_gif_path = None

    progress(0.98, desc="Packaging results")
    summary_csv = os.path.join(workdir, "cell_migration_summary.csv")
    pop_stats_csv = os.path.join(workdir, "population_stats_mean_sem.csv")
    tracks_csv = os.path.join(workdir, "tracked_cells_all_points.csv")
    drop_reasons_csv = os.path.join(workdir, "dropped_trajectories_reasons.csv")
    summary.to_csv(summary_csv, index=False)
    pop_stats_df.to_csv(pop_stats_csv, index=False)
    filtered.to_csv(tracks_csv, index=False)
    drop_reasons_df.to_csv(drop_reasons_csv, index=False)

    zip_path = os.path.join(workdir, "electrotaxis_results.zip")
    with zipfile.ZipFile(zip_path, "w") as zf:
        for p in [qc_path, traj_path, polar_path, box_path,
                  summary_csv, pop_stats_csv, tracks_csv, drop_reasons_csv]:
            zf.write(p, arcname=os.path.basename(p))
        if overlay_gif_path and os.path.exists(overlay_gif_path):
            zf.write(overlay_gif_path, arcname=os.path.basename(overlay_gif_path))

    n_total_linked = linked["cell_id"].nunique()
    n_kept = filtered["cell_id"].nunique()
    n_complete = filtered.loc[filtered["is_complete"], "cell_id"].nunique()
    frame_order_preview = ", ".join(frame_names[:3] + (["..."] if len(frame_names) > 3 else []) + frame_names[-1:] if len(frame_names) > 4 else frame_names)
    status = (
        f"**Done.** {n_frames} frames processed.\n\n"
        f"- Frame order used (check this matches time order!): {frame_order_preview}\n"
        f"- Raw linked trajectories: {n_total_linked}\n"
        f"- Kept after quality filtering: {n_kept}\n"
        f"- Complete trajectories (span all frames): {n_complete}\n"
        f"- Used for GPU: {'yes' if use_gpu else 'no'}\n\n"
        f"See the **Drop reasons** tab for why the {n_total_linked - n_kept} "
        f"non-kept trajectories were excluded, and the **Movie overlay** tab "
        f"to scrub through all {n_frames} frames with outlines/IDs/trajectories."
    )

    # Bundle what BMAC field-reversal analysis (a separate, optional, later
    # step) needs to re-derive an exact per-frame pixel mask for any single
    # tracked cell, without re-running segmentation/tracking.
    bmac_state = {
        "masks": masks,
        "images": images,
        # columns needed by BMAC (single-cell) AND the population-level
        # dynamics analyses (directional-cosine-over-time, population
        # area-response, field-of-view morphology overlay, interval analysis)
        "track_points": filtered[[
            "cell_id", "cell_number", "frame", "label",
            "x_um", "y_um", "time_min", "is_complete",
        ]].copy(),
        "frame_interval_min": frame_interval_min,
        "n_frames": n_frames,
        "cell_id_to_number": cell_id_to_number,
        "field_angle": field_angle,
    }
    cell_number_choices = [f"Cell {n}" for n in sorted(cell_id_to_number.values())]
    cell_dropdown_update = gr.update(
        choices=cell_number_choices,
        value=cell_number_choices[0] if cell_number_choices else None,
    )

    progress(1.0, desc="Done")
    return (
        status,
        qc_path, traj_path, polar_path, box_path,
        summary, pop_stats_df,
        drop_reasons_df, overlay_gif_path,
        zip_path,
        bmac_state, cell_dropdown_update,
    )


# ----------------------------------------------------------------------
# Full-movie overlay (outlines + centroid + ID + trailing trajectory),
# frame by frame, for eyeballing doubtful tracking/segmentation cases
# (merges, gaps, jumps) across the whole movie rather than just frame 0.
# ----------------------------------------------------------------------
_OVERLAY_CMAP = plt.get_cmap("tab20")


def _color_for_cell(cell_number):
    return _OVERLAY_CMAP((cell_number - 1) % 20)


def render_movie_overlay(images, masks, filtered, out_dir, trail_frames=10):
    """Render one PNG per frame: grayscale image + outline of each kept
    cell present in that frame (from its exact Cellpose label, so merged
    or mis-tracked masks are visible as-is, not hidden), its centroid,
    its cell_number, and a short trailing line of its last `trail_frames`
    positions. Returns the sorted list of frame image paths."""
    n_frames = len(images)
    h, w = images[0].shape[:2]

    # Pre-index for fast per-frame lookup.
    by_frame = {f: g for f, g in filtered.groupby("frame")}
    by_cell = {c: g.sort_values("frame") for c, g in filtered.groupby("cell_id")}

    paths = []
    for frame_idx in range(n_frames):
        fig, ax = plt.subplots(figsize=(w / 150, h / 150), dpi=150)
        ax.imshow(images[frame_idx], cmap="gray")

        frame_rows = by_frame.get(frame_idx)
        if frame_rows is not None:
            for _, row in frame_rows.iterrows():
                cell_id = row["cell_id"]
                cell_number = int(row["cell_number"])
                color = _color_for_cell(cell_number)
                label = int(row["label"])
                cell_mask = masks[frame_idx] == label

                for contour in find_contours(cell_mask.astype(float), 0.5):
                    ax.plot(contour[:, 1], contour[:, 0], linewidth=0.8, color=color)

                ax.plot(row["x"], row["y"], marker="+", markersize=6, color=color)
                ax.annotate(
                    str(cell_number), (row["x"], row["y"]),
                    xytext=(4, 4), textcoords="offset points",
                    color=color, fontsize=6, fontweight="bold",
                )

                track = by_cell[cell_id]
                trail = track[(track["frame"] <= frame_idx) &
                              (track["frame"] > frame_idx - trail_frames)]
                if len(trail) > 1:
                    ax.plot(trail["x"], trail["y"], linewidth=0.6, color=color, alpha=0.7)

        ax.set_title(f"Frame {frame_idx}", fontsize=8)
        ax.axis("off")
        frame_path = os.path.join(out_dir, f"frame_{frame_idx:04d}.png")
        fig.savefig(frame_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        paths.append(frame_path)
    return paths


# ----------------------------------------------------------------------
# BMAC electric-field reversal analysis (Sroka et al. 2018, sec. 3.5,
# points 5-9 and Fig. 5): track how much of a single cell's rightmost and
# leftmost thirds (relative to the field axis) grow or shrink around the
# moment the field is reversed.
# ----------------------------------------------------------------------
def _nearest_available_frame(track_points_for_cell, target_frame):
    """Return the row (as a dict) whose frame is closest to target_frame,
    or None if the cell has no tracked frames at all."""
    if len(track_points_for_cell) == 0:
        return None
    idx = (track_points_for_cell["frame"] - target_frame).abs().idxmin()
    return track_points_for_cell.loc[idx].to_dict()


def _cell_mask_at_frame(masks, track_points_for_cell, frame):
    """Boolean pixel mask for this cell at an EXACT frame, or None if the
    cell wasn't tracked in that frame (occlusion/gap)."""
    row = track_points_for_cell[track_points_for_cell["frame"] == frame]
    if len(row) == 0:
        return None
    label = int(row["label"].iloc[0])
    return masks[frame] == label


def _left_right_areas(cell_mask):
    """Split a cell's bounding box into left/center/right thirds along the
    x axis (the field axis, per the paper's chamber geometry) and return
    the pixel-area of the mask within the left and right thirds. Center
    third is excluded, matching the paper."""
    rows_idx, cols_idx = np.where(cell_mask)
    if len(cols_idx) == 0:
        return np.nan, np.nan
    col_min, col_max = cols_idx.min(), cols_idx.max()
    span = col_max - col_min
    if span == 0:
        return np.nan, np.nan
    third = span / 3.0
    left_edge = col_min + third
    right_edge = col_max - third
    area_left = int(np.sum(cols_idx < left_edge))
    area_right = int(np.sum(cols_idx > right_edge))
    return area_left, area_right


def run_bmac_analysis(
    state, selected_cell_label,
    reversal_time_min, baseline_before_min, duration_after_min,
    progress=gr.Progress(),
):
    if state is None:
        raise gr.Error("Run the main pipeline first — BMAC analysis reuses its segmentation results.")
    if not selected_cell_label:
        raise gr.Error("Pick a cell to analyze.")

    cell_number = int(str(selected_cell_label).replace("Cell", "").strip())
    cell_id_to_number = state["cell_id_to_number"]
    number_to_cell_id = {v: k for k, v in cell_id_to_number.items()}
    if cell_number not in number_to_cell_id:
        raise gr.Error(f"Cell {cell_number} isn't in the tracked results.")
    cell_id = number_to_cell_id[cell_number]

    masks = state["masks"]
    frame_interval_min = state["frame_interval_min"]
    n_frames = state["n_frames"]
    track_points = state["track_points"]
    track_points_cell = track_points[track_points["cell_id"] == cell_id]

    if len(track_points_cell) == 0:
        raise gr.Error(f"Cell {cell_number} has no tracked frames.")

    progress(0.1, desc="Locating reversal and baseline frames")
    reversal_frame_target = round(reversal_time_min / frame_interval_min)
    baseline_frame_target = round((reversal_time_min - baseline_before_min) / frame_interval_min)
    end_frame_target = round((reversal_time_min + duration_after_min) / frame_interval_min)

    reversal_frame_target = int(np.clip(reversal_frame_target, 0, n_frames - 1))
    baseline_frame_target = int(np.clip(baseline_frame_target, 0, n_frames - 1))
    end_frame_target = int(np.clip(end_frame_target, 0, n_frames - 1))

    reversal_row = _nearest_available_frame(track_points_cell, reversal_frame_target)
    baseline_row = _nearest_available_frame(track_points_cell, baseline_frame_target)
    if reversal_row is None or baseline_row is None:
        raise gr.Error(f"Cell {cell_number} wasn't tracked near the requested times.")
    reversal_frame = int(reversal_row["frame"])
    baseline_frame = int(baseline_row["frame"])

    progress(0.25, desc="Measuring left/right surface area per frame")
    frames_in_window = sorted(
        f for f in track_points_cell["frame"].unique()
        if baseline_frame_target <= f <= end_frame_target
    )
    if reversal_frame not in frames_in_window:
        frames_in_window = sorted(set(frames_in_window) | {reversal_frame})
    if baseline_frame not in frames_in_window:
        frames_in_window = sorted(set(frames_in_window) | {baseline_frame})

    baseline_mask = _cell_mask_at_frame(masks, track_points_cell, baseline_frame)
    area_left_0, area_right_0 = _left_right_areas(baseline_mask)

    rows = []
    for f in frames_in_window:
        cell_mask = _cell_mask_at_frame(masks, track_points_cell, f)
        area_left, area_right = _left_right_areas(cell_mask)
        s_left = ((area_left - area_left_0) / area_left_0) if area_left_0 else np.nan
        s_right = ((area_right - area_right_0) / area_right_0) if area_right_0 else np.nan
        rows.append({
            "frame": f,
            "time_min": f * frame_interval_min,
            "time_rel_to_reversal_min": (f - reversal_frame) * frame_interval_min,
            "area_left_px": area_left, "area_right_px": area_right,
            "S_left": s_left, "S_right": s_right,
        })
    area_df = pd.DataFrame(rows).sort_values("frame")

    workdir = tempfile.mkdtemp(prefix="bmac_")

    progress(0.6, desc="Plotting area change")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(area_df["time_rel_to_reversal_min"], area_df["S_left"],
            marker="o", label="S_left (anode-facing after reversal)", color="tab:red")
    ax.plot(area_df["time_rel_to_reversal_min"], area_df["S_right"],
            marker="o", label="S_right (cathode-facing after reversal)", color="tab:green")
    ax.axvline(0, color="black", linestyle="--", linewidth=1, label="Field reversal")
    ax.set_xlabel("Time relative to field reversal (min)")
    ax.set_ylabel("S = (S_t - S_0) / S_0")
    ax.set_title(f"Cell {cell_number}: surface area change around field reversal")
    ax.legend(fontsize=8)
    plot_path = os.path.join(workdir, f"cell{cell_number}_bmac_area_change.png")
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    progress(0.8, desc="Rendering expansion/retraction overlay")
    # Fig. 5b style: compare the cell's contour at reversal against snapshots
    # after reversal; pixels gained = green (protrusion), pixels lost = red
    # (retraction). Offsets are chosen in whole multiples of the movie's own
    # frame interval, so every panel is guaranteed to land on a genuinely
    # different frame -- fixed offsets like "+1 min" would silently collapse
    # onto the same frame (or the reversal frame itself) whenever the frame
    # interval is coarser than that, e.g. a 5 min/frame movie.
    max_k = int(np.floor(duration_after_min / frame_interval_min)) if frame_interval_min > 0 else 0
    if max_k >= 1:
        all_ks = list(range(1, max_k + 1))
        if len(all_ks) > 4:
            # cap at 4 panels, evenly spaced across the available range
            pick_idx = np.linspace(0, len(all_ks) - 1, 4).round().astype(int)
            chosen_ks = sorted(set(all_ks[i] for i in pick_idx))
        else:
            chosen_ks = all_ks
        candidate_offsets = [k * frame_interval_min for k in chosen_ks]
    else:
        # duration_after_min is smaller than one frame interval -- nothing
        # after reversal is actually available; fall back to whatever was
        # asked for so the error/empty-panel messaging below still applies.
        candidate_offsets = [duration_after_min] if duration_after_min > 0 else []

    reversal_mask = _cell_mask_at_frame(masks, track_points_cell, reversal_frame)
    n_panels = max(1, len(candidate_offsets))
    fig2, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels, 4))
    if n_panels == 1:
        axes = [axes]

    overlay_path = os.path.join(workdir, f"cell{cell_number}_bmac_overlay.png")
    if reversal_mask is None or not candidate_offsets:
        for ax2 in axes:
            ax2.text(0.5, 0.5, "Not enough tracked frames\nfor an overlay",
                      ha="center", va="center")
            ax2.axis("off")
    else:
        rows_r, cols_r = np.where(reversal_mask)
        pad = 15
        r0, r1 = max(rows_r.min() - pad, 0), rows_r.max() + pad
        c0, c1 = max(cols_r.min() - pad, 0), cols_r.max() + pad

        for ax2, offset_min in zip(axes, candidate_offsets):
            target_frame_target = reversal_frame + round(offset_min / frame_interval_min)
            target_row = _nearest_available_frame(track_points_cell, target_frame_target)
            target_mask = _cell_mask_at_frame(masks, track_points_cell, int(target_row["frame"])) \
                if target_row is not None else None

            crop_shape = (r1 - r0, c1 - c0, 3)
            canvas = np.ones(crop_shape) * 0.9  # light gray background
            if target_mask is not None:
                green = target_mask & ~reversal_mask
                red = reversal_mask & ~target_mask
                unchanged = reversal_mask & target_mask
                canvas[unchanged[r0:r1, c0:c1]] = [0.5, 0.5, 0.5]
                canvas[green[r0:r1, c0:c1]] = [0.1, 0.8, 0.1]
                canvas[red[r0:r1, c0:c1]] = [0.9, 0.1, 0.1]
            ax2.imshow(canvas)
            ax2.set_title(f"+{offset_min:g} min")
            ax2.axis("off")
    fig2.suptitle(f"Cell {cell_number}: green = expansion, red = retraction (vs. reversal frame)")
    fig2.savefig(overlay_path, dpi=200, bbox_inches="tight")
    plt.close(fig2)

    progress(0.95, desc="Saving data")
    csv_path = os.path.join(workdir, f"cell{cell_number}_bmac_area_change.csv")
    area_df.to_csv(csv_path, index=False)

    status = (
        f"**Cell {cell_number} — BMAC field-reversal analysis done.**\n\n"
        f"- Baseline frame: {baseline_frame} ({baseline_frame * frame_interval_min:.1f} min)\n"
        f"- Reversal frame used: {reversal_frame} ({reversal_frame * frame_interval_min:.1f} min)\n"
        f"- Frames analyzed: {len(area_df)}\n"
    )
    if baseline_frame != baseline_frame_target or reversal_frame != reversal_frame_target:
        status += "- Note: cell wasn't tracked at exactly the requested time(s); nearest available frame was used.\n"

    progress(1.0, desc="Done")
    return status, plot_path, overlay_path, csv_path


# ----------------------------------------------------------------------
# Population-level dynamics analyses, following Lasota et al. 2024
# (BBA Mol. Cell Res. 1871:119647), Figs. 2-4.
# ----------------------------------------------------------------------
def _parse_reversal_times(reversal_times_str):
    if not reversal_times_str or not str(reversal_times_str).strip():
        return []
    out = []
    for part in str(reversal_times_str).split(","):
        part = part.strip()
        if part:
            out.append(float(part))
    return out


def run_directional_dynamics(
    state, bin_min, reversal_times_str,
    progress=gr.Progress(),
):
    """Fig. 2 style: population mean +/- SEM directional cosine, binned
    every `bin_min` minutes across the whole recording, referenced to the
    FIXED field axis set in Calibration (not re-oriented at a reversal) --
    so a later field reversal shows up as the curve crossing toward
    negative values, exactly as in the paper."""
    if state is None:
        raise gr.Error("Run the main pipeline first.")

    track = state["track_points"]
    field_angle = state["field_angle"]
    frame_interval_min = state["frame_interval_min"]
    n_frames = state["n_frames"]
    if bin_min <= 0:
        raise gr.Error("Bin size must be greater than 0.")

    progress(0.2, desc="Binning trajectories")
    bin_frames = max(1, round(bin_min / frame_interval_min))
    edges = list(range(0, n_frames, bin_frames))
    if edges[-1] != n_frames - 1:
        edges.append(n_frames - 1)

    rows = []
    for f0, f1 in zip(edges[:-1], edges[1:]):
        cos_vals = []
        for cell_id, cell_track in track.groupby("cell_id"):
            row0 = _nearest_available_frame(cell_track, f0)
            row1 = _nearest_available_frame(cell_track, f1)
            if row0 is None or row1 is None:
                continue
            # skip if the nearest available frame is too far from the
            # requested bin edge (cell not really present in this bin)
            if abs(row0["frame"] - f0) > bin_frames or abs(row1["frame"] - f1) > bin_frames:
                continue
            dx = row1["x_um"] - row0["x_um"]
            dy = row1["y_um"] - row0["y_um"]
            if dx == 0 and dy == 0:
                continue
            theta = np.arctan2(dy, dx)
            cos_vals.append(np.cos(theta - field_angle))
        m, se, n = mean_sem_n(pd.Series(cos_vals))
        rows.append({
            "bin_start_min": f0 * frame_interval_min,
            "bin_end_min": f1 * frame_interval_min,
            "bin_mid_min": (f0 + f1) / 2 * frame_interval_min,
            "mean_cos_gamma": m, "sem": se, "n": n,
        })
    dyn_df = pd.DataFrame(rows)

    progress(0.7, desc="Plotting")
    workdir = tempfile.mkdtemp(prefix="dynamics_")
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(
        dyn_df["bin_mid_min"], dyn_df["mean_cos_gamma"], yerr=dyn_df["sem"],
        marker="o", markersize=4, capsize=3, color="tab:red",
    )
    ax.axhline(0, color="gray", linewidth=0.8)
    for rt in _parse_reversal_times(reversal_times_str):
        ax.axvline(rt, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Directional cos γ (mean ± SEM)")
    ax.set_title(f"Directional cosine dynamics (bin = {bin_min:g} min)")
    plot_path = os.path.join(workdir, "directional_dynamics.png")
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    csv_path = os.path.join(workdir, "directional_dynamics.csv")
    dyn_df.to_csv(csv_path, index=False)

    status = f"**Done.** {len(dyn_df)} time bins computed across {n_frames} frames."
    progress(1.0, desc="Done")
    return status, plot_path, csv_path


def _combined_fov_mask(masks, track_at_frame, image_shape):
    """Union of every kept cell's pixel mask at one frame, built from their
    tracked labels (not the raw Cellpose mask, so untracked/filtered-out
    objects are excluded)."""
    combined = np.zeros(image_shape, dtype=bool)
    if len(track_at_frame) == 0:
        return combined
    frame_idx = int(track_at_frame["frame"].iloc[0])
    frame_mask = masks[frame_idx]
    labels = track_at_frame["label"].astype(int).unique()
    combined = np.isin(frame_mask, labels)
    return combined


def run_fov_morphology_overlay(state, event_time_min, offsets_str, progress=gr.Progress()):
    """Fig. 3 style: whole-field-of-view green/red change maps between
    consecutive timepoints (given as offsets in minutes relative to
    `event_time_min`, e.g. an EF application or reversal moment), using
    every kept cell at once rather than a single selected cell."""
    if state is None:
        raise gr.Error("Run the main pipeline first.")

    offsets = _parse_reversal_times(offsets_str)
    if len(offsets) < 2:
        raise gr.Error("Give at least two comma-separated offsets, e.g. -10,0,2,4")
    offsets = sorted(offsets)

    track = state["track_points"]
    masks = state["masks"]
    frame_interval_min = state["frame_interval_min"]
    n_frames = state["n_frames"]
    image_shape = state["images"][0].shape[:2]

    progress(0.2, desc="Locating frames")
    frame_targets = []
    for off in offsets:
        t = event_time_min + off
        f = int(np.clip(round(t / frame_interval_min), 0, n_frames - 1))
        frame_targets.append(f)

    workdir = tempfile.mkdtemp(prefix="fov_overlay_")
    n_panels = len(frame_targets) - 1
    fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels, 4.5))
    if n_panels == 1:
        axes = [axes]

    progress(0.5, desc="Rendering panels")
    for i, (fA, fB, offA, offB) in enumerate(
        zip(frame_targets[:-1], frame_targets[1:], offsets[:-1], offsets[1:])
    ):
        track_a = track[track["frame"] == fA]
        track_b = track[track["frame"] == fB]
        mask_a = _combined_fov_mask(masks, track_a, image_shape)
        mask_b = _combined_fov_mask(masks, track_b, image_shape)

        canvas = np.ones((*image_shape, 3)) * 0.95
        unchanged = mask_a & mask_b
        green = mask_b & ~mask_a
        red = mask_a & ~mask_b
        canvas[unchanged] = [0.5, 0.5, 0.5]
        canvas[green] = [0.1, 0.8, 0.1]
        canvas[red] = [0.9, 0.1, 0.1]

        axes[i].imshow(canvas)
        axes[i].set_title(f"{offA:g} to {offB:g} min")
        axes[i].axis("off")

    fig.suptitle(
        f"Field-of-view morphology change (green = protrusion, red = retraction)\n"
        f"relative to event at t={event_time_min:g} min"
    )
    overlay_path = os.path.join(workdir, "fov_morphology_overlay.png")
    fig.savefig(overlay_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    status = f"**Done.** {n_panels} panel(s) rendered from {len(offsets)} timepoints."
    progress(1.0, desc="Done")
    return status, overlay_path


def run_population_area_dynamics(
    state, event_time_min, baseline_before_min, duration_after_min,
    bin_min, complete_only,
    progress=gr.Progress(),
):
    """Fig. 4 style: population-averaged (mean +/- SEM across cells)
    S_left/S_right relative-area response around an event (field
    application or reversal), generalizing the single-cell BMAC analysis
    to every kept cell."""
    if state is None:
        raise gr.Error("Run the main pipeline first.")

    track = state["track_points"]
    masks = state["masks"]
    frame_interval_min = state["frame_interval_min"]
    n_frames = state["n_frames"]
    if bin_min <= 0:
        raise gr.Error("Time bin must be greater than 0.")

    if complete_only:
        track = track[track["is_complete"]]
    if track["cell_id"].nunique() == 0:
        raise gr.Error("No cells available for this analysis (try unchecking 'complete trajectories only').")

    progress(0.1, desc="Locating baseline / event frames")
    baseline_target = int(np.clip(round((event_time_min - baseline_before_min) / frame_interval_min), 0, n_frames - 1))
    event_target = int(np.clip(round(event_time_min / frame_interval_min), 0, n_frames - 1))
    end_target = int(np.clip(round((event_time_min + duration_after_min) / frame_interval_min), 0, n_frames - 1))

    bin_frames = max(1, round(bin_min / frame_interval_min))
    sample_frames = sorted(set(range(baseline_target, end_target + 1, bin_frames)) | {event_target, baseline_target, end_target})
    sample_frames = [f for f in sample_frames if baseline_target <= f <= end_target]

    progress(0.3, desc="Measuring per-cell left/right areas")
    per_cell_series = {}  # cell_id -> {frame: (S_left, S_right)}
    for cell_id, cell_track in track.groupby("cell_id"):
        baseline_row = _nearest_available_frame(cell_track, baseline_target)
        if baseline_row is None:
            continue
        baseline_mask = _cell_mask_at_frame(masks, cell_track, int(baseline_row["frame"]))
        area_left_0, area_right_0 = _left_right_areas(baseline_mask)
        if not area_left_0 or not area_right_0:
            continue

        series = {}
        for f in sample_frames:
            cell_mask = _cell_mask_at_frame(masks, cell_track, f)
            if cell_mask is None:
                continue
            area_left, area_right = _left_right_areas(cell_mask)
            if area_left is None or np.isnan(area_left):
                continue
            s_left = (area_left - area_left_0) / area_left_0
            s_right = (area_right - area_right_0) / area_right_0
            series[f] = (s_left, s_right)
        if series:
            per_cell_series[cell_id] = series

    if not per_cell_series:
        raise gr.Error("No cells had usable frames in this window.")

    progress(0.6, desc="Averaging across cells")
    rows = []
    for f in sample_frames:
        s_lefts = [v[f][0] for v in per_cell_series.values() if f in v]
        s_rights = [v[f][1] for v in per_cell_series.values() if f in v]
        m_left, se_left, n_left = mean_sem_n(pd.Series(s_lefts))
        m_right, se_right, n_right = mean_sem_n(pd.Series(s_rights))
        rows.append({
            "frame": f,
            "time_rel_to_event_min": (f - event_target) * frame_interval_min,
            "S_left_mean": m_left, "S_left_sem": se_left, "n_left": n_left,
            "S_right_mean": m_right, "S_right_sem": se_right, "n_right": n_right,
        })
    pop_area_df = pd.DataFrame(rows).sort_values("time_rel_to_event_min")

    progress(0.85, desc="Plotting")
    workdir = tempfile.mkdtemp(prefix="pop_area_")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.errorbar(
        pop_area_df["time_rel_to_event_min"], pop_area_df["S_left_mean"],
        yerr=pop_area_df["S_left_sem"], marker="o", markersize=4, capsize=3,
        color="tab:red", label="Left side (S_left)",
    )
    ax.errorbar(
        pop_area_df["time_rel_to_event_min"], pop_area_df["S_right_mean"],
        yerr=pop_area_df["S_right_sem"], marker="o", markersize=4, capsize=3,
        color="tab:green", label="Right side (S_right)",
    )
    ax.axvline(0, color="black", linestyle="--", linewidth=1, label="Event")
    ax.set_xlabel("Time relative to event (min)")
    ax.set_ylabel("Relative area change, S = (S_t - S_0) / S_0")
    n_cells_used = len(per_cell_series)
    ax.set_title(f"Population morphology response (n = {n_cells_used} cells)")
    ax.legend(fontsize=8)
    plot_path = os.path.join(workdir, "population_area_dynamics.png")
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    csv_path = os.path.join(workdir, "population_area_dynamics.csv")
    pop_area_df.to_csv(csv_path, index=False)

    status = f"**Done.** Averaged over {n_cells_used} cells, {len(pop_area_df)} time points."
    progress(1.0, desc="Done")
    return status, plot_path, csv_path


# ----------------------------------------------------------------------
# Cumulative-window analysis: the same per-cell parameters, recomputed using
# only the data from a reference start time (default 0) up to each chosen
# end time T (0->100 min, 0->200 min, ...), to see how they evolve.
# ----------------------------------------------------------------------
ALLOWED_GAP_FRAMES = 2  # a cell may be missing up to this many frames at a window edge

WINDOW_PARAMS = [
    ("Displacement_um", "Displacement (µm)"),
    ("Velocity_um_min", "Velocity (µm/min)"),
    ("Angle_deg", "Angle to field (deg, circular mean)"),
    ("Cos_angle", "Net cos θ"),
    ("Speed_um_min", "Speed (µm/min)"),
    ("Displacement_along_field_um", "Displacement along field (µm)"),
    ("Total_distance_um", "Total distance (µm)"),
]


def _window_metrics(frames, xs, ys, f_start, f_end, dt_frame, field_angle):
    """Same parameters as the main pipeline, but using only the part of one
    cell's track inside [f_start, f_end]. Returns None if the cell does not
    cover the window (so every cell in a window spans the SAME time range)."""
    m = (frames >= f_start) & (frames <= f_end)
    if m.sum() < 2:
        return None
    fr, x, y = frames[m], xs[m], ys[m]
    if fr[0] > f_start + ALLOWED_GAP_FRAMES or fr[-1] < f_end - ALLOWED_GAP_FRAMES:
        return None
    duration = (fr[-1] - fr[0]) * dt_frame
    if duration <= 0:
        return None
    dx, dy = x[-1] - x[0], y[-1] - y[0]
    disp = float(np.hypot(dx, dy))
    path = float(np.hypot(np.diff(x), np.diff(y)).sum())
    if disp > 0:
        theta = float(_wrap_angle(np.arctan2(dy, dx) - field_angle))
        angle_deg, cos_angle = float(np.degrees(theta)), float(np.cos(theta))
        along = disp * cos_angle
    else:
        angle_deg = cos_angle = np.nan
        along = 0.0
    return {
        "Displacement_um": disp,
        "Velocity_um_min": disp / duration,
        "Angle_deg": angle_deg,
        "Cos_angle": cos_angle,
        "Speed_um_min": path / duration,
        "Displacement_along_field_um": along,
        "Total_distance_um": path,
    }


def run_window_analysis(
    state, start_min, end_times_str, auto_step_min, complete_only,
    progress=gr.Progress(),
):
    if state is None:
        raise gr.Error("Run the main pipeline first.")

    track = state["track_points"]
    if complete_only:
        track = track[track["is_complete"]]
    if track["cell_id"].nunique() == 0:
        raise gr.Error("No cells available (try unchecking 'complete trajectories only').")

    field_angle = state["field_angle"]
    dt_frame = state["frame_interval_min"]
    n_frames = state["n_frames"]
    total_min = (n_frames - 1) * dt_frame
    start_min = float(start_min or 0)

    # End times: the typed list, or multiples of the auto step.
    end_times = _parse_reversal_times(end_times_str)
    if not end_times:
        if not auto_step_min or auto_step_min <= 0:
            raise gr.Error("Give window end times (e.g. 100, 200, 300) or an auto step in minutes.")
        end_times = list(np.arange(start_min + auto_step_min, total_min + 1e-9, auto_step_min))
        if not end_times or end_times[-1] < total_min - 1e-9:
            end_times.append(total_min)
    end_times = sorted({round(min(t, total_min), 6) for t in end_times if t > start_min})
    if not end_times:
        raise gr.Error("All end times are at or before the start time.")

    progress(0.1, desc="Preparing trajectories")
    cells = {}
    for cid, g in track.groupby("cell_id"):
        g = g.sort_values("frame")
        cells[cid] = (
            int(g["cell_number"].iloc[0]),
            g["frame"].to_numpy(), g["x_um"].to_numpy(), g["y_um"].to_numpy(),
        )

    progress(0.3, desc="Computing parameters for each window")
    f_start = int(round(start_min / dt_frame))
    rows = []
    for T in end_times:
        f_end = int(round(T / dt_frame))
        if f_end <= f_start:
            continue
        for cid, (cnum, frames, xs, ys) in cells.items():
            rec = _window_metrics(frames, xs, ys, f_start, f_end, dt_frame, field_angle)
            if rec is None:
                continue
            rec.update({"cell_id": cid, "cell_number": cnum,
                        "window_start_min": start_min, "window_end_min": T})
            rows.append(rec)
    if not rows:
        raise gr.Error("No cell covers any of the requested windows — try shorter windows "
                       "or uncheck 'complete trajectories only'.")
    per_cell_df = pd.DataFrame(rows)

    progress(0.6, desc="Averaging across cells")
    pop_rows = []
    for (t0, T), g in per_cell_df.groupby(["window_start_min", "window_end_min"]):
        row = {"window_start_min": t0, "window_end_min": T, "n_cells": len(g)}
        for col, _ in WINDOW_PARAMS:
            if col == "Angle_deg":
                a = np.radians(g[col].dropna().to_numpy())
                row[f"{col}_mean"] = (float(np.degrees(np.arctan2(np.mean(np.sin(a)), np.mean(np.cos(a)))))
                                      if len(a) else np.nan)
                row[f"{col}_sem"] = np.nan
            else:
                m, se, _n = mean_sem_n(g[col])
                row[f"{col}_mean"], row[f"{col}_sem"] = m, se
        # what unbiased random motion would give for the mean cos (95% band)
        row["Cos_angle_random_95ci"] = 1.96 * np.sqrt(0.5) / np.sqrt(len(g))
        pop_rows.append(row)
    pop_df = pd.DataFrame(pop_rows).sort_values("window_end_min")

    progress(0.8, desc="Plotting")
    workdir = tempfile.mkdtemp(prefix="windows_")
    x = pop_df["window_end_min"]
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)
    axes = axes.ravel()
    for ax, (col, label) in zip(axes, WINDOW_PARAMS):
        if col == "Angle_deg":
            ax.plot(x, pop_df[f"{col}_mean"], marker="o", ms=4, color="tab:blue")
            ax.set_ylim(-185, 185)
            ax.axhline(0, color="gray", linewidth=0.8)
        else:
            ax.errorbar(x, pop_df[f"{col}_mean"], yerr=pop_df[f"{col}_sem"],
                        marker="o", ms=4, capsize=3, color="tab:blue")
        if col == "Cos_angle":
            ax.fill_between(x, -pop_df["Cos_angle_random_95ci"], pop_df["Cos_angle_random_95ci"],
                            color="gray", alpha=0.2, label="random-motion 95% band")
            ax.axhline(0, color="gray", linewidth=0.8)
            ax.set_ylim(-1.05, 1.05)
            ax.legend(fontsize=7)
        if col == "Displacement_along_field_um":
            ax.axhline(0, color="gray", linewidth=0.8)
        ax.set_title(label, fontsize=10)
        ax.set_xlabel(f"Window end T (min), window = {start_min:g} → T")
    for ax in axes[len(WINDOW_PARAMS):]:
        ax.axis("off")
    # n per window, as text in the spare panel
    spare = axes[len(WINDOW_PARAMS)]
    spare.axis("on")
    spare.plot(x, pop_df["n_cells"], marker="s", ms=4, color="tab:gray")
    spare.set_title("Cells contributing (n)", fontsize=10)
    spare.set_xlabel(f"Window end T (min), window = {start_min:g} → T")
    fig.suptitle("Parameters vs. analysis window (mean ± SEM across cells)")
    fig.tight_layout()
    plot_path = os.path.join(workdir, "window_analysis.png")
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    progress(0.95, desc="Saving data")
    paths = []
    for name, df in [("window_per_cell.csv", per_cell_df), ("window_population.csv", pop_df)]:
        p = os.path.join(workdir, name)
        df.to_csv(p, index=False)
        paths.append(p)

    n_excluded_note = ""
    n_all = len(cells)
    min_n, max_n = int(pop_df["n_cells"].min()), int(pop_df["n_cells"].max())
    if min_n < n_all:
        n_excluded_note = (f"\n\nOnly cells tracked across the whole window are used, so n goes "
                           f"from {max_n} down to {min_n} (of {n_all} kept cells). Check the "
                           f"n panel before comparing windows.")
    status = f"**Done.** {len(pop_df)} windows ({start_min:g} → T)." + n_excluded_note
    progress(1.0, desc="Done")
    return status, plot_path, pop_df, paths


# ----------------------------------------------------------------------
# UI
# ----------------------------------------------------------------------
with gr.Blocks(title="Electrotaxis Cell Tracking") as demo:
    gr.Markdown(
        "# Electrotaxis Cell Tracking\n"
        "Upload every TIFF frame of your movie, set the parameters for your "
        "experiment, and run the pipeline. Segmentation (Cellpose) → linking "
        "(trackpy) → quality filtering → displacement / velocity / angle metrics → averages "
        "(mean ± SEM)."
        + ("\n\n_Running on a shared ZeroGPU Space — GPU is allocated only "
           "while segmentation runs._" if HAS_ZEROGPU else "")
    )

    with gr.Row():
        with gr.Column(scale=1):
            files_in = gr.Files(label="TIFF frames (select all frames of the movie)", file_types=[".tif", ".tiff"])

            with gr.Accordion("Calibration", open=True):
                frame_interval_in = gr.Number(value=1.0, label="Frame interval (min)")
                pixel_to_um_in = gr.Number(value=0.86959, label="Pixel size (µm/pixel)")
                field_x_in = gr.Number(value=1.0, label="Electric field direction, x")
                field_y_in = gr.Number(value=0.0, label="Electric field direction, y")

            with gr.Accordion("Tracking (trackpy)", open=False):
                search_range_in = gr.Number(value=50, label="Search range (px)")
                memory_in = gr.Number(value=2, precision=0, label="Memory (frames)")

            with gr.Accordion("Quality filters", open=False):
                min_len_in = gr.Number(value=10, precision=0, label="Min track length (frames)")
                edge_margin_in = gr.Number(value=15, label="Edge margin (px)")
                max_jump_in = gr.Number(value=1.8, label="Max frame-to-frame area ratio (division)")
                min_ratio_in = gr.Number(value=0.5, label="Min frame-to-frame area ratio (rounding/death)")
                require_complete_in = gr.Checkbox(value=True, label="Require complete trajectories for the averages")

            use_gpu_in = gr.Checkbox(value=True, label="Use GPU for segmentation (falls back to CPU if unavailable)")
            run_btn = gr.Button("Run pipeline", variant="primary")

        with gr.Column(scale=1):
            status_out = gr.Markdown()
            with gr.Tab("QC image"):
                qc_out = gr.Image(label="Cells included, frame 0")
            with gr.Tab("Trajectories"):
                traj_out = gr.Image(label="Trajectories (relative coordinates)")
            with gr.Tab("Polar plot"):
                polar_out = gr.Image(label="Distance vs. direction")
            with gr.Tab("Speed distribution"):
                box_out = gr.Image(label="Speed boxplot")
            with gr.Tab("Drop reasons"):
                drop_reasons_out = gr.Dataframe(
                    label="Why trajectories were kept or dropped (quality filtering)",
                    wrap=True,
                )
            with gr.Tab("Movie overlay"):
                gr.Markdown(
                    "Every frame with cell outlines, centroid, ID, and a short "
                    "trailing trajectory — scrub through to check doubtful cases "
                    "(merges, gaps, jumps) across the whole movie."
                )
                movie_overlay_out = gr.Image(label="Movie overlay (animated GIF)")

    with gr.Row():
        summary_out = gr.Dataframe(label="Per-cell summary", wrap=True)
    with gr.Row():
        pop_stats_out = gr.Dataframe(label="Population stats (mean ± SEM)", wrap=True)

    zip_out = gr.File(label="Download all results (.zip)")

    pipeline_state = gr.State()  # holds masks/images/track points for BMAC analysis

    gr.Markdown("## Additional analysis")
    bmac_yes_in = gr.Radio(
        ["No", "Yes"], value="No",
        label="Perform BMAC return analysis (electric-field reversal)?",
    )
    with gr.Column(visible=False) as bmac_options:
        gr.Markdown(
            "Cell numbers match the labels shown in the **QC image** tab above "
            "(run the pipeline first, then come back here)."
        )
        bmac_cell_in = gr.Dropdown(label="Which cell?", choices=[])
        bmac_reversal_time_in = gr.Number(
            label="When should the field be reversed? (minutes from movie start)"
        )
        bmac_baseline_in = gr.Number(
            value=5, label="Baseline window before reversal (minutes)"
        )
        bmac_duration_in = gr.Number(
            value=5, label="Analysis duration after reversal (minutes)"
        )
        bmac_run_btn = gr.Button("Run BMAC reversal analysis", variant="secondary")
        bmac_status_out = gr.Markdown()
        bmac_plot_out = gr.Image(label="Surface-area change (S_left / S_right) vs. time")
        bmac_overlay_out = gr.Image(label="Expansion (green) / retraction (red) vs. reversal frame")
        bmac_csv_out = gr.File(label="Download area-change data (.csv)")

    bmac_yes_in.change(
        fn=lambda choice: gr.update(visible=(choice == "Yes")),
        inputs=bmac_yes_in, outputs=bmac_options,
    )

    run_btn.click(
        fn=run_pipeline,
        inputs=[
            files_in,
            frame_interval_in, pixel_to_um_in,
            field_x_in, field_y_in,
            search_range_in, memory_in,
            min_len_in, edge_margin_in,
            max_jump_in, min_ratio_in,
            require_complete_in, use_gpu_in,
        ],
        outputs=[
            status_out,
            qc_out, traj_out, polar_out, box_out,
            summary_out, pop_stats_out,
            drop_reasons_out, movie_overlay_out,
            zip_out,
            pipeline_state, bmac_cell_in,
        ],
    )

    bmac_run_btn.click(
        fn=run_bmac_analysis,
        inputs=[
            pipeline_state, bmac_cell_in,
            bmac_reversal_time_in, bmac_baseline_in, bmac_duration_in,
        ],
        outputs=[bmac_status_out, bmac_plot_out, bmac_overlay_out, bmac_csv_out],
    )

    gr.Markdown(
        "## Population dynamics analyses\n"
        "_Following Lasota et al. 2024 (BBA Mol. Cell Res. 1871:119647), "
        "Figs. 2–4 — reaction speed of the whole tracked population to "
        "field application or reversal. These reuse the same run above, "
        "no need to re-upload or re-segment._"
    )

    with gr.Accordion("Directional cosine dynamics over time (Fig. 2)", open=False):
        gr.Markdown(
            "Population mean ± SEM directional cosine, binned over time across "
            "the whole recording. Uses the **fixed field axis** set in "
            "Calibration above — so if you list a reversal time, the curve "
            "should visibly cross toward negative values there, not just "
            "flatten out, since the reference axis doesn't rotate with the field."
        )
        dyn_bin_in = gr.Number(value=5, label="Time bin (minutes)")
        dyn_reversals_in = gr.Textbox(
            label="Field reversal time(s), comma-separated (minutes; leave blank if none)",
            placeholder="e.g. 120  or  60, 180",
        )
        dyn_run_btn = gr.Button("Compute directional dynamics", variant="secondary")
        dyn_status_out = gr.Markdown()
        dyn_plot_out = gr.Image(label="Directional cosine vs. time")
        dyn_csv_out = gr.File(label="Download data (.csv)")

    with gr.Accordion("Population morphology response to an event (Fig. 4)", open=False):
        gr.Markdown(
            "S_left / S_right relative area change, averaged across every "
            "kept cell (not just one), around a chosen event — field "
            "application (event time ≈ 0) or a later reversal."
        )
        pop_event_time_in = gr.Number(label="Event time (min from movie start)")
        pop_baseline_in = gr.Number(value=5, label="Baseline window before event (min)")
        pop_duration_in = gr.Number(value=10, label="Duration after event (min)")
        pop_bin_in = gr.Number(value=0.5, label="Time bin (minutes; paper used 30s = 0.5 min)")
        pop_complete_only_in = gr.Checkbox(value=False, label="Use complete trajectories only")
        pop_run_btn = gr.Button("Compute population morphology response", variant="secondary")
        pop_status_out = gr.Markdown()
        pop_plot_out = gr.Image(label="S_left / S_right vs. time (population mean ± SEM)")
        pop_csv_out = gr.File(label="Download data (.csv)")

    with gr.Accordion("Field-of-view morphology overlay (Fig. 3)", open=False):
        gr.Markdown(
            "Whole-field green/red change map between chosen timepoints, "
            "using every kept cell at once — reproduces the paper's "
            "sequential protrusion/retraction panels without picking a "
            "single cell."
        )
        fov_event_time_in = gr.Number(label="Reference event time (min from movie start)")
        fov_offsets_in = gr.Textbox(
            label="Panel timepoints, comma-separated offsets from the event (minutes)",
            placeholder="e.g. -10, 0, 2, 4",
        )
        fov_run_btn = gr.Button("Render field-of-view overlay", variant="secondary")
        fov_status_out = gr.Markdown()
        fov_overlay_out = gr.Image(label="Whole-field protrusion (green) / retraction (red)")

    with gr.Accordion("Parameters vs. analysis window (0 → T)", open=False):
        gr.Markdown(
            "Recomputes displacement, velocity, angle, cos θ, speed, displacement "
            "along the field and total distance using **only the data from the "
            "start time up to each end time T** (e.g. 0→100, 0→200 min), then "
            "shows how the population averages change with T. Only cells tracked "
            "across the whole window are used, so every window compares the same "
            "time span."
        )
        win_start_in = gr.Number(value=0, label="Reference start time (min)")
        win_ends_in = gr.Textbox(
            label="Window end times T, comma-separated (min)",
            placeholder="e.g. 100, 200, 300",
        )
        win_step_in = gr.Number(
            value=0, label="…or auto step (min), used only if the list above is blank",
        )
        win_complete_in = gr.Checkbox(value=False, label="Use complete trajectories only")
        win_run_btn = gr.Button("Compute window analysis", variant="secondary")
        win_status_out = gr.Markdown()
        win_plot_out = gr.Image(label="Parameters vs. window end time")
        win_pop_out = gr.Dataframe(label="Population averages per window", wrap=True)
        win_files_out = gr.Files(label="Download data (.csv)")

    dyn_run_btn.click(
        fn=run_directional_dynamics,
        inputs=[pipeline_state, dyn_bin_in, dyn_reversals_in],
        outputs=[dyn_status_out, dyn_plot_out, dyn_csv_out],
    )
    pop_run_btn.click(
        fn=run_population_area_dynamics,
        inputs=[
            pipeline_state, pop_event_time_in, pop_baseline_in,
            pop_duration_in, pop_bin_in, pop_complete_only_in,
        ],
        outputs=[pop_status_out, pop_plot_out, pop_csv_out],
    )
    fov_run_btn.click(
        fn=run_fov_morphology_overlay,
        inputs=[pipeline_state, fov_event_time_in, fov_offsets_in],
        outputs=[fov_status_out, fov_overlay_out],
    )

    win_run_btn.click(
        fn=run_window_analysis,
        inputs=[pipeline_state, win_start_in, win_ends_in, win_step_in, win_complete_in],
        outputs=[win_status_out, win_plot_out, win_pop_out, win_files_out],
    )

# Colab: launch directly (no __main__ guard) with a public link
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b357276f05a27fa43e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


100%|██████████| 1.15G/1.15G [00:09<00:00, 125MB/s]
/usr/local/lib/python3.13/dist-packages/cellpose/dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),
